<a href="https://colab.research.google.com/github/spoorthyhm494-stack/RAG/blob/main/Rag1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers faiss-cpu rank_bm25 pypdf groq arxiv scikit-learn matplotlib numpy
print("✅ Installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 10.8 MB/s eta 0:00:00
✅ Installed.


In [ ]:
import os
import re
import textwrap
import numpy as np
from getpass import getpass

import faiss                                    # fast vector search
from sentence_transformers import SentenceTransformer   # turns text into vectors
import arxiv                                     # download real papers
import requests
from pypdf import PdfReader                      # read PDF text
from groq import Groq                            # talk to the LLM

print("Imports ready.")

Imports ready.


In [ ]:
os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ")

client = Groq()
MODEL = "llama3-8b-8192"

def ask_llm(prompt: str, system: str = "", max_tokens: int = 600) -> str:
    """
    Sends one message to the LLM and returns its reply as plain text.
    Every other cell in this notebook calls this ONE function whenever it needs the LLM to think —
    that's the only place the model is actually called.
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=max_tokens,
        temperature=0,          # 0 = give the most likely answer, not a random/creative one
        messages=messages,
    )
    return response.choices[0].message.content

# quick test
print(ask_llm("Reply with exactly: LLM connection OK"))

Enter your GROQ_API_KEY: ··········


BadRequestError: Error code: 400 - {'error': {'message': 'The model `llama3-8b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

In [ ]:
import os
from getpass import getpass
from groq import Groq

# ==========================================
# 1. Enter your Groq API key
# ==========================================
os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ")

# ==========================================
# 2. Initialize Groq client
# ==========================================
client = Groq(api_key=os.environ["GROQ_API_KEY"])

# ==========================================
# 3. Use a specific model
# ==========================================
MODEL = "openai/gpt-oss-20b"

print("Using Model:", MODEL)

# ==========================================
# 4. Function to ask the LLM
# ==========================================
def ask_llm(prompt: str, system: str = "", max_tokens: int = 600) -> str:

    messages = []

    if system:
        messages.append({
            "role": "system",
            "content": system
        })

    messages.append({
        "role": "user",
        "content": prompt
    })

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=max_tokens,
        temperature=0
    )

    return response.choices[0].message.content


# ==========================================
# 5. Quick test
# ==========================================
print(ask_llm("Reply with exactly: LLM connection OK"))

Enter your GROQ_API_KEY: ··········
Using Model: openai/gpt-oss-20b
LLM connection OK


In [ ]:
# The exact papers we want, by their official arXiv ID.
PAPERS = [
    {"id": "1706.03762", "label": "Attention Is All You Need (the original Transformer paper)"},
    {"id": "2005.11401", "label": "Retrieval-Augmented Generation (the original RAG paper)"},
    {"id": "2210.03629", "label": "ReAct (reasoning + acting in AI agents)"},
]

os.makedirs("papers", exist_ok=True)
paper_files = []   # will hold {id, title, path} for each downloaded paper

client_arxiv = arxiv.Client()
for p in PAPERS:
    search = arxiv.Search(id_list=[p["id"]])
    result = next(client_arxiv.results(search))

    filepath = f"papers/{result.get_short_id()}.pdf"
    pdf_bytes = requests.get(result.pdf_url, timeout=30).content
    with open(filepath, "wb") as f:
        f.write(pdf_bytes)

    paper_files.append({"id": result.get_short_id(), "title": result.title, "path": filepath})
    print(f"Downloaded: {result.title}")

print(f"\n {len(paper_files)} papers saved to ./papers/")

Downloaded: Attention Is All You Need
Downloaded: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks
Downloaded: ReAct: Synergizing Reasoning and Acting in Language Models

 3 papers saved to ./papers/


In [ ]:
def get_pdf_text(path: str) -> str:
    """Reads every page of a PDF and joins the text together."""
    reader = PdfReader(path)
    pages = [page.extract_text() or "" for page in reader.pages]
    text = "\n".join(pages)
    text = re.sub(r"\s+", " ", text)   # collapse messy whitespace into single spaces
    return text

def split_into_chunks(text: str, chunk_size: int = 800, overlap: int = 100):
    """
    Cuts text into overlapping chunks of `chunk_size` characters.
    `overlap` characters are repeated at the start of the next chunk, so an idea that
    spans a chunk boundary doesn't get lost.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# Build our full list of chunks, across all 3 papers
all_chunks = []   # each item: {"id": ..., "title": ..., "text": ...}

for paper in paper_files:
    full_text = get_pdf_text(paper["path"])
    text_chunks = split_into_chunks(full_text)
    for i, chunk_text in enumerate(text_chunks):
        all_chunks.append({
            "id": f"{paper['id']}_chunk{i}",
            "title": paper["title"],
            "text": chunk_text,
        })

print(f" Created {len(all_chunks)} chunks from {len(paper_files)} papers")
print("\nExample chunk:")
print(textwrap.fill(all_chunks[10]["text"][:400], 100))

 Created 314 chunks from 3 papers

Example chunk:
regressive [10], consuming the previously generated symbols as additional input when generating the
next. 2 Figure 1: The Transformer - model architecture. The Transformer follows this overall
architecture using stacked self-attention and point-wise, fully connected layers for both the
encoder and decoder, shown in the left and right halves of Figure 1, respectively. 3.1 Encoder and
Decoder Stacks


In [ ]:
print("Loading the embedding model (turns text into vectors)...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Turn every chunk's text into a vector
chunk_texts = [c["text"] for c in all_chunks]
vectors = embed_model.encode(chunk_texts, show_progress_bar=True, normalize_embeddings=True)
vectors = np.array(vectors, dtype="float32")

print(f"\nEach chunk became a vector with {vectors.shape[1]} numbers.")
print(f"Example — first 8 numbers of chunk 0's vector: {vectors[0][:8]}")

# Store all vectors in a FAISS index for fast searching
index = faiss.IndexFlatIP(vectors.shape[1])   # IP = inner product; with normalized vectors this IS cosine similarity
index.add(vectors)

print(f"\n {index.ntotal} vectors stored and ready to search.")

Loading the embedding model (turns text into vectors)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]


Each chunk became a vector with 384 numbers.
Example — first 8 numbers of chunk 0's vector: [-0.07872316 -0.1392612   0.05405427 -0.02420468  0.03912175  0.06441186
 -0.09939577  0.00142398]

 314 vectors stored and ready to search.


In [ ]:
def find_relevant_chunks(question: str, top_k: int = 4):
    """Returns the top_k chunks whose meaning is closest to the question."""
    question_vector = embed_model.encode([question], normalize_embeddings=True)
    similarities, indices = index.search(np.array(question_vector, dtype="float32"), top_k)

    results = []
    for idx, score in zip(indices[0], similarities[0]):
        results.append({"chunk": all_chunks[idx], "similarity": float(score)})
    return results

# Try it
test_question = "What is positional encoding used for?"
matches = find_relevant_chunks(test_question)

print(f"Question: {test_question}\n")
for m in matches:
    print(f"[similarity = {m['similarity']:.3f}]  {m['chunk']['id']}  ({m['chunk']['title'][:40]}...)")
    print(textwrap.fill(m["chunk"]["text"][:220], 100))
    print()

Question: What is positional encoding used for?

[similarity = 0.509]  1706.03762v7_chunk23  (Attention Is All You Need...)
lative positions, since for any fixed offsetk,PE pos+k can be represented as a linear function of PE
pos. We also experimented with using learned positional embeddings [9] instead, and found that the
two versions produce

[similarity = 0.493]  1706.03762v7_chunk22  (Attention Is All You Need...)
ngs" to the input embeddings at the bottoms of the encoder and decoder stacks. The positional
encodings have the same dimensiondmodel as the embeddings, so that the two can be summed. There are
many choices of positional

[similarity = 0.480]  1706.03762v7_chunk12  (Attention Is All You Need...)
lso modify the self-attention sub-layer in the decoder stack to prevent positions from attending to
subsequent positions. This masking, combined with fact that the output embeddings are offset by one
position, ensures th

[similarity = 0.419]  1706.03762v7_chunk11  (Attention Is Al

In [ ]:
ANSWER_INSTRUCTIONS = (
    "You are a helpful research assistant. Answer the question using ONLY the "
    "context provided below. For every fact you state, mention which chunk it came from, like [chunk_id]. "
    "If the context does not contain the answer, say so clearly instead of guessing."
)

def answer_question(question: str, top_k: int = 4):
    matches = find_relevant_chunks(question, top_k=top_k)

    # Build the context block the LLM will read
    context_text = ""
    for m in matches:
        c = m["chunk"]
        context_text += f"\n[{c['id']}] (from: {c['title']})\n{c['text']}\n"

    prompt = f"Context:\n{context_text}\n\nQuestion: {question}\n\nAnswer:"
    answer = ask_llm(prompt, system=ANSWER_INSTRUCTIONS, max_tokens=400)
    return answer, matches

# Try the full pipeline end to end
question = "What is positional encoding used for?"
answer, sources = answer_question(question)

print("QUESTION:", question)
print("\nANSWER:")
print(textwrap.fill(answer, 100))
print("\nSOURCES USED:", [m["chunk"]["id"] for m in sources])

QUESTION: What is positional encoding used for?

ANSWER:


SOURCES USED: ['1706.03762v7_chunk23', '1706.03762v7_chunk22', '1706.03762v7_chunk12', '1706.03762v7_chunk11']


In [ ]:
questions_to_try = [
    "Why is attention called 'scaled dot-product attention'?",
    "What is the ReAct pattern for AI agents?",
    "What is the capital of France?",
]

for q in questions_to_try:
    print("="*90)
    print("QUESTION:", q)
    answer, sources = answer_question(q)
    print("\nANSWER:")
    print(textwrap.fill(answer, 100))
    print("\nSOURCES:", [m["chunk"]["id"] for m in sources])
    print()

QUESTION: Why is attention called 'scaled dot-product attention'?

ANSWER:
Attention is called **“scaled dot‑product attention”** because the core operation is a dot product
between the query and key vectors, and this dot product is **scaled** by a factor of
\(1/\sqrt{d_k}\) before the softmax is applied. The scaling is introduced to keep the dot‑product
values from becoming too large (which would push the softmax into a region with very small
gradients) [chunk13]. The name reflects both the use of the dot product and the scaling step that
precedes the softmax.

SOURCES: ['1706.03762v7_chunk14', '1706.03762v7_chunk15', '1706.03762v7_chunk13', '1706.03762v7_chunk12']

QUESTION: What is the ReAct pattern for AI agents?

ANSWER:
The **ReAct pattern** is a closed‑loop framework that lets an AI agent interleave *reasoning* and
*acting* within a single policy.   Key elements of the pattern are:  1. **Augmented action space** –
the agent’s action set is expanded to      \(\hat{A}=A\cup L\), w

In [24]:
!git clone https://github.com/spoorthyhm494-stack/RAG

Cloning into 'RAG'...


In [28]:
%cd RAG

/content/RAG


In [34]:
!ls /content

papers	RAG  sample_data
